# Detect spots: .ome.tiff -> H5 -> ilastik

In [ ]:
import subprocess
from pathlib import Path

import h5py
import numpy as np
import vigra
from aicsimageio import AICSImage

input_dir = Path("/srv/scratch/berrylab/z3536241/NikonSpinningDisk/260515_mACPOLR2A_CRC_EU")
tmp_dir = Path("/srv/scratch/berrylab/z3532965/tmp")
tmp_dir.mkdir(parents=True, exist_ok=True)

ilastik_path = "/srv/scratch/berrylab/z3532965/ilastik-1.4.1-Linux/run_ilastik.sh"
ilastik_project = "/srv/scratch/berrylab/z3532965/spot_classifier.ilp"

channels = [1, 2]

In [ ]:
files = sorted(input_dir.glob("*.ome.tiff"))
print(f"Found {len(files)} .ome.tiff files")
assert files, "No .ome.tiff files found"

In [ ]:
axistags = vigra.defaultAxistags("tzyxc")
h5_paths = []

for f in files:
    img = AICSImage(f)
    data = img.get_image_data("CYX", T=0, Z=0)  # CYX
    data = data[channels, :, :]  # keep channels 1 and 2

    h5_path = tmp_dir / (f.name.replace(".ome.tiff", "") + ".h5")
    image_yxc = np.transpose(data, (1, 2, 0))  # CYX -> YXC
    image_5d = image_yxc[np.newaxis, np.newaxis, :, :, :]  # TZYXC

    with h5py.File(h5_path, "w") as h5_file:
        ds = h5_file.create_dataset(
            name="data", data=image_5d, chunks=(1, 1, 64, 64, 1)
        )
        ds.attrs["axistags"] = axistags.toJSON()
    h5_paths.append(str(h5_path))

print(f"Wrote {len(h5_paths)} H5 files to {tmp_dir}")

In [ ]:
subprocess.run(
    [ilastik_path, "--headless", "--project", ilastik_project] + h5_paths,
    check=True,
)
print("Ilastik processing complete")

In [ ]:
subprocess.run(["rm"] + h5_paths)
print("Removed temporary H5 files")